# EvidenceLab #11 | 49 VARIABLES. 4 SELECTED. 3 SPATIALLY EXPLORED.
## **Clean the data. Select the signals. Prepare for spatial discovery.**

**Dr. Amobi Andrew Onovo, PhD, MPH** | EvidenceLab  
**See it. Understand it. Run it.**

> **Beginner-friendly, end to end:** No uploads, no account keys, no external files. Select **Runtime → Run all** in [Google Colab](https://colab.research.google.com/). All teaching data are generated within this notebook.

### What is new in this expanded edition?
**Start with data quality, not a model.** We intentionally generate realistic data problems, then investigate **missing values, duplicates, negative values, out-of-range entries, and outliers**. Every chart in this notebook uses **interactive Plotly**, so you can hover for exact values, zoom, pan, hide traces, and export charts with the Plotly toolbar. Some charts also offer variable selection through a dropdown menu.

**Research inspiration, not data replication:** Onovo A. and colleagues, *Mapping the Future: Leveraging Lasso Regression and GWR to Guide HIV Testing and Prevention Strategies in Tanzania*, **AIDS 2024, poster TUPED353**. The authors analyzed 49 candidate variables across Tanzania's 31 regions using the 2022–23 Tanzania HIV Impact Survey and the 2022 Tanzania DHS. The original poster reported four variables selected by both CV and minBIC Lasso, then three mapped by geographically weighted regression (GWR) after excluding one for multicollinearity.

[Read the original poster via Quantium Insights Research](https://quantiuminsights.com/research.html) → **Presentations, awards and funding**.

> **Critical scientific distinction:** The numbers and interactive plots you compute here are *entirely synthetic*. They must not be described as Tanzania estimates, model validation of the poster, or causal effects. The poster's adjusted GWR R² = 0.784 is an in-sample measure of fit, **not 78.4% prediction accuracy**. Its discussion uses “HIV acquisition,” while some figures refer to prevalence: these require distinct outcome definitions in new research.

## Roadmap: The Dr. Onovo Pathway

**Lesson 10:** Triangulate and harmonize diverse data sources.  
**Lesson 11:** **AUDIT → CLEAN → VALIDATE → LASSO-SELECT** indicators.  
**Lesson 12:** Examine spatial variation with GWR and uncertainty-aware regional interpretation.

| Stage | Public health question | What we do today |
|---|---|---|
| Define | What is the outcome and who is represented? | Use a synthetic continuous prevalence-like outcome and fictional regions |
| Audit | Are the data trustworthy enough to analyze? | Visualize missingness, duplicates, invalid negatives, range violations and outliers |
| Treat | Which defects can be resolved, and how? | Deduplicate with an explicit rule; set implausible values to missing; impute and cap within CV |
| Select | Which variables contribute under regularization? | Grouped cross-validation, Lasso penalties and stability checks |
| Localize | Do associations change across space? | Prepare a fictional GWR handoff file; GWR is covered in Lesson 12 |

**Learning outcome:** You will understand *why* each data-quality decision was made, not merely how to run a line of code.

**Important:** Lasso screens *statistical associations*, not causation. GWR can explore *where associations vary* but is not proof that a mapped variable caused an outbreak or any individual infection.

## 0. RUN: Set up your laboratory

**Step-by-step**

1. Open this `.ipynb` file in Google Colab.
2. Choose **Runtime → Run all**; the first code cell imports the required libraries.
3. Explore the charts with hover, zoom, legend toggles and dropdowns.
4. Use **File → Save a copy in Drive** only if you want to keep your own work.

**What you should see:** A short readiness message. Plotly, pandas, NumPy and scikit-learn are supported in Colab. If Plotly is absent in a custom environment, the setup cell installs it automatically; there is still no dataset download or upload.

In [1]:
import sys, subprocess, importlib.util, warnings
if importlib.util.find_spec('plotly') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'plotly>=5.18'])

from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display, Markdown
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 202411
rng = np.random.default_rng(SEED)
BLUE, DEEP, TEAL, GOLD, CORAL = '#004A7C', '#0D174F', '#00BFA6', '#F0B429', '#F05A28'
COLORS = [BLUE, TEAL, CORAL, '#7643BA', GOLD, '#18A0C9', '#D9425F']

# Interactive Plotly template, consistent across all illustrations.
custom = go.layout.Template(pio.templates['plotly_white'])
custom.layout.update(
    font=dict(family='Arial, sans-serif', color=DEEP, size=13),
    paper_bgcolor='white', plot_bgcolor='#F5F9FC',
    colorway=COLORS,
    title=dict(font=dict(size=19, color=DEEP)),
    margin=dict(l=55, r=35, t=85, b=65),
    hoverlabel=dict(bgcolor='white', font=dict(color=DEEP, size=12))
)
pio.templates['evidencelab'] = custom
px.defaults.template = 'evidencelab'
try:
    import google.colab
    pio.renderers.default = 'colab'
except ImportError:
    pio.renderers.default = 'plotly_mimetype'

# Editable lesson controls. Re-run from the top after changing them.
USE_TRAINING_ONLY_OUTLIER_CAPPING = True
LOWER_CAP_QUANTILE, UPPER_CAP_QUANTILE = 0.01, 0.99
print(f'✔ Ready | Python {sys.version.split()[0]} | Plotly {__import__("plotly").__version__}')
print('✔ All data are created locally. All visualizations use interactive Plotly.')
print('✔ No matplotlib, static charts, file uploads, or API keys required.')

✔ Ready | Python 3.13.5 | Plotly 6.5.2
✔ All data are created locally. All visualizations use interactive Plotly.
✔ No matplotlib, static charts, file uploads, or API keys required.


# PART A | Prepare a deliberately imperfect dataset

## 1. CREATE fictional public health data

**Research question:** From 49 candidate indicators, which variables are most predictive of differences in a continuous, synthetic health outcome across fictional areas, and how can we prepare for a later spatial analysis?

We construct **31 fictional regions × 12 fictional service areas = 372 original records**. The first four predictors use familiar names from the research poster; another 45 are generic teaching indicators. We deliberately program four underlying simulated associations into the outcome. These are not observed Tanzania data.

**Why deliberately add errors?** Real datasets often contain wrong entries, missing information and duplicate exports. A notebook containing perfect data cannot teach data cleaning.

In [2]:
N_REGIONS, AREAS_PER_REGION = 31, 12
N = N_REGIONS * AREAS_PER_REGION
region_num = np.repeat(np.arange(N_REGIONS), AREAS_PER_REGION)
region_labels = np.array([f'Fictional_region_{r+1:02d}' for r in region_num])
service_ids = [f'A{r+1:02d}_{a+1:02d}' for r in range(N_REGIONS) for a in range(AREAS_PER_REGION)]

latent = rng.normal(size=(N, 4))
features = pd.DataFrame({
 'men_alcohol_use_pct': np.clip(21 + 7*latent[:,0], 1, 95),
 'women_primary_education_pct': np.clip(43 + 12*latent[:,1], 1, 98),
 'women_age_at_first_sex_years': np.clip(17 + 1.1*latent[:,2], 12, 24),
 'women_hiv_results_received_pct': np.clip(61 + 9*latent[:,3], 1, 98)
})
TRUE_SYNTHETIC_SIGNALS = list(features.columns)
for j in range(45):
    features[f'other_indicator_{j+1:02d}'] = rng.normal(size=N)
PREDICTORS = list(features.columns)

region_effect = rng.normal(0, .48, size=N_REGIONS)
outcome = (14 + 2.0*latent[:,0] - 1.8*latent[:,1] + 1.3*latent[:,2]
           - 1.7*latent[:,3] + region_effect[region_num] + rng.normal(0,1.15,size=N))
outcome = np.clip(outcome, 0.5, 37)

clean_source = features.copy()
clean_source.insert(0, 'service_area_id', service_ids)
clean_source.insert(0, 'region_id', region_labels)
clean_source['recorded_at'] = '2025-04-01'
clean_source['fictional_x'] = 8*(region_num % 7) + rng.normal(0, .9, N)
clean_source['fictional_y'] = 8*(region_num // 7) + rng.normal(0, .9, N)
clean_source['synthetic_prevalence_pct'] = outcome

# Inject errors into a COPY. The clean reference is not used to repair the data.
dirty = clean_source.copy(deep=True)
for column, how_many in [('men_alcohol_use_pct',16),
                         ('women_primary_education_pct',13),
                         ('women_age_at_first_sex_years',10),
                         ('women_hiv_results_received_pct',17),
                         ('other_indicator_03',18),('other_indicator_07',12),
                         ('other_indicator_11',9)]:
    affected = rng.choice(N, size=how_many, replace=False)
    dirty.loc[affected, column] = np.nan

# Simulated non-numeric input errors.
dirty['women_primary_education_pct'] = dirty['women_primary_education_pct'].astype(object)
dirty.loc[[12, 43], 'women_primary_education_pct'] = 'not recorded'
# Negative values are impossible for percentages and age; signed generic indicators may be negative.
dirty.loc[[6, 92, 164], 'men_alcohol_use_pct'] = [-7, -3, -12]
dirty.loc[[38, 221], 'women_age_at_first_sex_years'] = [-4, -2]
dirty.loc[[52, 209], 'women_hiv_results_received_pct'] = [121, 135]
dirty.loc[[78], 'women_primary_education_pct'] = [107]
dirty.loc[[160], 'women_age_at_first_sex_years'] = [88]  # clearly outside teaching range
# Extreme but potentially plausible in a signed generic feature: flag, do not delete.
dirty.loc[[5, 105, 275], 'other_indicator_01'] = [11.5, -12.0, 13.2]
dirty.loc[[27, 137, 242], 'synthetic_prevalence_pct'] = np.nan
dirty.loc[[33, 155], 'synthetic_prevalence_pct'] = [-2, 118]

# Five exact copies + three later versions with an edited measure.
exact_copies = dirty.iloc[[2, 51, 111, 171, 301]].copy()
later_versions = dirty.iloc[[17, 59, 103]].copy()
later_versions['recorded_at'] = '2025-05-01'
later_versions['other_indicator_04'] += 0.7
raw_data = pd.concat([dirty, exact_copies, later_versions], ignore_index=True)
raw_snapshot = raw_data.copy(deep=True)  # preserved for audit and reproducibility
assert len(PREDICTORS) == 49 and len(clean_source) == 372
print(f'Created {len(raw_snapshot)} raw rows, {len(PREDICTORS)} candidate predictors, {N_REGIONS} fictional regions.')
print('Expected: missing entries, invalid values, outliers, 5 exact duplicates and 3 competing record versions.')
display(raw_snapshot[['region_id','service_area_id','men_alcohol_use_pct',
                      'women_primary_education_pct','synthetic_prevalence_pct']].head(7))

Created 380 raw rows, 49 candidate predictors, 31 fictional regions.
Expected: missing entries, invalid values, outliers, 5 exact duplicates and 3 competing record versions.


,region_id,service_area_id,men_alcohol_use_pct,women_primary_education_pct,synthetic_prevalence_pct
0,Fictional_region_01,A01_01,25.317176,62.101357,8.727973
1,Fictional_region_01,A01_02,20.410276,52.229673,11.425529
2,Fictional_region_01,A01_03,30.783647,44.055183,21.724001
3,Fictional_region_01,A01_04,34.018627,42.686411,17.512227
4,Fictional_region_01,A01_05,26.980193,16.754623,16.653096
5,Fictional_region_01,A01_06,NaN,56.384367,16.297484
6,Fictional_region_01,A01_07,-7.000000,27.510775,17.633398


## 2. AUDIT A: Data structure and missing values

**Definitions for beginners:**

- **Missing data:** A value is unknown or not supplied. Do not assume it equals zero.
- **Blank or coded as text:** A cell saying `not recorded` also represents missing data, even though it is not technically `NaN` until converted.
- **Predictor versus outcome:** You may impute missing *predictors* using training data. Do not fabricate missing *outcomes* just to keep rows.

**Exercise:** Before running, predict whether the dataset has exactly 372 rows. Why might it be larger? Run the audit below and hover on each bar.

In [3]:
work = raw_snapshot.copy(deep=True)
# Explicit parsing: bad strings become NaN; keep the raw snapshot untouched.
conversion_failures = {}
for col in PREDICTORS + ['synthetic_prevalence_pct','fictional_x','fictional_y']:
    before = int(work[col].notna().sum())
    work[col] = pd.to_numeric(work[col], errors='coerce')
    conversion_failures[col] = before - int(work[col].notna().sum())

missing_count = work[PREDICTORS + ['synthetic_prevalence_pct']].isna().sum()
missing_rate = (100*missing_count/len(work)).round(2)
missing_table = pd.DataFrame({'variable':missing_count.index,
    'missing_n':missing_count.values,'missing_pct':missing_rate.values})
missing_table = missing_table.sort_values('missing_n', ascending=False)

print(f'Rows: {len(work)} | Predictors: {len(PREDICTORS)} | Unique regions: {work.region_id.nunique()}')
print(f'Empty predictor cells: {int(work[PREDICTORS].isna().sum().sum())}')
print(f'Non-numeric entries converted to missing: {sum(conversion_failures.values())}')
display(missing_table.head(12))

nonzero = missing_table.query('missing_n > 0')
fig = go.Figure()
fig.add_bar(x=missing_table.variable, y=missing_table.missing_pct,
            marker_color=BLUE, name='All 50 analysis fields',
            customdata=missing_table.missing_n,
            hovertemplate='%{x}<br>Missing: %{customdata} rows (%{y:.1f}%)<extra></extra>')
fig.add_bar(x=nonzero.variable, y=nonzero.missing_pct,
            marker_color=CORAL, name='Variables with missingness', visible=False,
            customdata=nonzero.missing_n,
            hovertemplate='%{x}<br>Missing: %{customdata} rows (%{y:.1f}%)<extra></extra>')
fig.update_layout(
 title='A1. Missingness by field | Choose a view',
 xaxis_title='Candidate indicators and outcome', yaxis_title='Missing cells (%)',
 xaxis_tickangle=-65, height=560, showlegend=False,
 updatemenus=[dict(type='dropdown', direction='down', x=1, y=1.21,
   buttons=[dict(label='All 50 analysis fields', method='update',
                 args=[{'visible':[True,False]}]),
            dict(label='Only fields with missing entries', method='update',
                 args=[{'visible':[False,True]}])])]
)
fig.show()
print('HOW TO READ: A taller bar means a higher share of missing values, not higher disease risk.')
print('ACTION: Examine the missingness mechanism. Later, numeric predictors use training-fold median imputation.')

Rows: 380 | Predictors: 49 | Unique regions: 31
Empty predictor cells: 100
Non-numeric entries converted to missing: 2


,variable,missing_n,missing_pct
6,other_indicator_03,19,5.00
3,women_hiv_results_received_pct,17,4.47
1,women_primary_education_pct,16,4.21
0,men_alcohol_use_pct,16,4.21
10,other_indicator_07,12,3.16
2,women_age_at_first_sex_years,10,2.63
14,other_indicator_11,10,2.63
49,synthetic_prevalence_pct,3,0.79
5,other_indicator_02,0,0.00
7,other_indicator_04,0,0.00


HOW TO READ: A taller bar means a higher share of missing values, not higher disease risk.
ACTION: Examine the missingness mechanism. Later, numeric predictors use training-fold median imputation.


## 3. AUDIT B: Interactive missing-data map

A missingness heatmap shows *where missing cells occur*. Each horizontal line is a raw record and each column is a variable. Use hover to examine individual cells.

**Interpretation prompt:** Are missing values clustered in specific predictors or scattered across records? In real surveys you would also evaluate whether missingness varies by site, region, sex or time.

In [4]:
heat_columns = list(missing_table.head(16)['variable'])
heat = work[heat_columns].isna().astype(int).T
fig = go.Figure(go.Heatmap(z=heat.to_numpy(), x=np.arange(len(work)), y=heat.index,
 colorscale=[[0.0,'#E8F5FA'],[0.499,'#E8F5FA'],[0.5,'#F05A28'],[1.0,'#F05A28']],
 zmin=0,zmax=1, showscale=False,
 hovertemplate='Variable: %{y}<br>Raw record index: %{x}<br>Missing flag: %{z}<extra></extra>'))
fig.update_layout(title='A2. Missing-value map | 0 = present, 1 = missing',
    xaxis_title='Raw record index (includes duplicates)', yaxis_title='', height=600)
fig.show()
print('INTERPRET: Orange cells are missing. This map displays the 16 fields with most missing cells.')

INTERPRET: Orange cells are missing. This map displays the 16 fields with most missing cells.


## 4. AUDIT C: Duplicate records

**Two distinct problems:**

1. **Exact duplicates:** Every field matches an earlier row. The extra copy can usually be removed after checking import history.
2. **Competing versions:** The *same* region and service-area ID appears more than once but some fields differ. You need a documented source/version rule, not an arbitrary `drop_duplicates()` call.

**Our synthetic rule:** Remove exact copies first. For competing versions, keep the latest `recorded_at` entry. In a real epidemiological study, verify that date and provenance reflect authoritative updates before choosing a record.

In [5]:
KEYS = ['region_id','service_area_id']
exact_duplicate_count = int(work.duplicated().sum())
key_duplicate_count = int(work.duplicated(subset=KEYS,keep=False).sum())
conflict_keys = (work.drop_duplicates()
                   .groupby(KEYS, dropna=False).size().gt(1).sum())
qc_dup = pd.DataFrame({'check':['Extra exact-copy rows','Rows sharing an area key','Area keys with competing versions'],
                       'count':[exact_duplicate_count,key_duplicate_count,int(conflict_keys)]})
fig=px.bar(qc_dup, x='check', y='count', color='check',
           color_discrete_sequence=COLORS, text='count',
           title='A3. Duplicate audit | Exact and competing versions are different issues')
fig.update_traces(textposition='outside',hovertemplate='%{x}<br>%{y} flagged<extra></extra>')
fig.update_layout(showlegend=False, height=430, xaxis_title='', yaxis_title='Number')
fig.show()
print(f'Exact extra copies = {exact_duplicate_count}. Area IDs with competing versions = {int(conflict_keys)}.')
print('ACTION: Keep the raw snapshot, remove exact copies, then resolve competing versions by recorded date.')

Exact extra copies = 5. Area IDs with competing versions = 3.
ACTION: Keep the raw snapshot, remove exact copies, then resolve competing versions by recorded date.


## 5. AUDIT D: Negative numbers and domain validation

**A negative number is not automatically an error.** A signed residual, growth measure or standardized variable may legitimately be negative. A **percentage** or **age** cannot be negative, and a percent cannot exceed 100.

Our teaching constraints:

| Variable type | Valid illustrative range | Treatment outside range |
|---|---:|---|
| Percent indicators | 0 to 100 | Set as missing and investigate source |
| Age at first sex (synthetic demonstration) | 12 to 30 years | Set as missing for this exercise only |
| Synthetic outcome (%) | 0 to 100 | Exclude invalid outcome rows after version resolution |
| Generic signed indicators | No artificial non-negative rule | Preserve negatives; inspect extreme values separately |

**Caution:** The 12–30 years age rule is a *teaching assumption*, not a universal exclusion standard. Real data cleaning requires a codebook, subject-matter expertise, sensitivity analysis and attention to potentially valid edge cases.

In [6]:
BOUNDS = {
 'men_alcohol_use_pct': (0,100),
 'women_primary_education_pct':(0,100),
 'women_hiv_results_received_pct':(0,100),
 'women_age_at_first_sex_years':(12,30),
 'synthetic_prevalence_pct':(0,100)
}
invalid_rows = []
for col,(low,high) in BOUNDS.items():
    s=work[col]
    invalid_rows.append({'variable':col,'negative_n':int((s<0).sum()),
                         'above_max_n':int((s>high).sum()),
                         'below_min_n':int((s<low).sum())})
invalid_table = pd.DataFrame(invalid_rows)
long = invalid_table.melt(id_vars='variable',value_vars=['negative_n','above_max_n'],
                          var_name='issue',value_name='count')
fig=px.bar(long,x='variable',y='count',color='issue',barmode='group',
 title='A4. Domain-rule violations | Values requiring investigation',
 labels={'variable':'Constrained field','count':'Invalid entries','issue':'Problem type'},
 color_discrete_map={'negative_n':CORAL,'above_max_n':'#7C3BB7'})
fig.update_layout(height=490,xaxis_tickangle=-22)
fig.show()
print(f'Generic signed variable other_indicator_02 has {int((work.other_indicator_02<0).sum())} negative values.')
print('INTERPRET: Those signed predictor negatives are not invalid. Apply domain rules, not blanket deletion.')
display(invalid_table)

Generic signed variable other_indicator_02 has 192 negative values.
INTERPRET: Those signed predictor negatives are not invalid. Apply domain rules, not blanket deletion.


,variable,negative_n,above_max_n,below_min_n
0,men_alcohol_use_pct,3,0,3
1,women_primary_education_pct,0,1,0
2,women_hiv_results_received_pct,0,2,0
3,women_age_at_first_sex_years,2,1,2
4,synthetic_prevalence_pct,1,1,1


## 6. AUDIT E: Outlier detection and responsible treatment

**An outlier is not always an error.** It may indicate a data-entry mistake, a true but uncommon setting, or the early signal of a public health problem.

- Use the familiar **interquartile range** (IQR) rule to flag suspicious extremes for review: outside Q1 − 1.5×IQR to Q3 + 1.5×IQR.
- Do **not** automatically delete all flagged observations.
- Our synthetic impossible values become `NaN` using explicit bounds; later, plausible extreme *predictors* can be gently capped with limits learned **inside training folds** to reduce leverage.

**Interactive exercise:** Select a variable in the dropdown. Hover over the points on the box plot and bins in the histogram.

In [7]:
OUTLIER_VIEW = [
 'men_alcohol_use_pct','women_primary_education_pct',
 'women_age_at_first_sex_years','women_hiv_results_received_pct','other_indicator_01'
]
fig = make_subplots(rows=1,cols=2,
 subplot_titles=('Distribution of the selected variable','Box plot and flagged extremes'),
 column_widths=[.65,.35],horizontal_spacing=.13)
for j,col in enumerate(OUTLIER_VIEW):
    values=work[col].dropna().astype(float)
    fig.add_trace(go.Histogram(x=values,nbinsx=28,marker_color=BLUE,
         name=col,showlegend=False, visible=(j==0),
         hovertemplate='Value bin: %{x}<br>Count: %{y}<extra></extra>'),row=1,col=1)
    fig.add_trace(go.Box(y=values,boxpoints='outliers',marker_color=CORAL,
         line_color=BLUE,name=col,showlegend=False,visible=(j==0),
         hovertemplate='Value: %{y:.2f}<extra></extra>'),row=1,col=2)
buttons=[]
for j,col in enumerate(OUTLIER_VIEW):
    visibility=[False]*(2*len(OUTLIER_VIEW))
    visibility[2*j]=visibility[2*j+1]=True
    buttons.append(dict(label=col, method='update',args=[{'visible':visibility},
        {'title':f'A5. Outlier explorer | {col}'}]))
fig.update_layout(height=520,title=f'A5. Outlier explorer | {OUTLIER_VIEW[0]}',
 updatemenus=[dict(type='dropdown',x=.99,y=1.25,direction='down',buttons=buttons)],
 bargap=.05)
fig.update_xaxes(title_text='Value',row=1,col=1)
fig.update_yaxes(title_text='Records',row=1,col=1)
fig.update_yaxes(title_text='Value',row=1,col=2)
fig.show()

flags=[]
for col in OUTLIER_VIEW:
    s=work[col].dropna().astype(float)
    q1,q3=s.quantile([.25,.75]); iqr=q3-q1
    flags.append({'variable':col,'IQR_outlier_flags':int(((s<q1-1.5*iqr)|(s>q3+1.5*iqr)).sum()),
                 'valid_observations':len(s)})
display(pd.DataFrame(flags))
print('INTERPRET: A box-plot point is a flag for examination, not a license to discard it.')

,variable,IQR_outlier_flags,valid_observations
0,men_alcohol_use_pct,9,364
1,women_primary_education_pct,5,364
2,women_age_at_first_sex_years,8,370
3,women_hiv_results_received_pct,2,363
4,other_indicator_01,5,380


INTERPRET: A box-plot point is a flag for examination, not a license to discard it.


# PART B | CLEAN using explicit, reproducible decisions

## 7. Resolve duplicates and invalid values, without hiding what happened

**Read this before running:** We preserve `raw_snapshot` unchanged. We remove exact copies, use the most recent dated record for conflicting area IDs, convert impossible predictor values to missing, and drop rows with unknown/invalid **outcomes**. We do **not** fill missing predictors with whole-dataset medians. That would leak information across our held-out regions.

**Ask yourself:** If two conflicting records have no trustworthy timestamps, what additional evidence would you need before choosing one?

In [8]:
# 7A. Deterministic record-level cleaning (does not estimate model parameters).
cleaned = work.drop_duplicates().copy()
cleaned['recorded_at'] = pd.to_datetime(cleaned['recorded_at'], errors='coerce')
assert cleaned['recorded_at'].notna().all(), 'Unusable record timestamps. Check version-resolution rule.'
cleaned = cleaned.sort_values(KEYS + ['recorded_at'], ascending=[True,True,False])
cleaned = cleaned.drop_duplicates(subset=KEYS, keep='first').sort_values(KEYS).reset_index(drop=True)

invalid_counter = {}
for col,(lo,hi) in BOUNDS.items():
    mask=cleaned[col].notna() & ((cleaned[col]<lo)|(cleaned[col]>hi))
    invalid_counter[col] = int(mask.sum())
    cleaned.loc[mask,col]=np.nan

invalid_target_rows = int(cleaned['synthetic_prevalence_pct'].isna().sum())
analysis_data = cleaned.loc[cleaned['synthetic_prevalence_pct'].notna()].reset_index(drop=True).copy()
assert analysis_data.duplicated(subset=KEYS).sum()==0
assert analysis_data['synthetic_prevalence_pct'].between(0,100).all()
assert len(analysis_data) > 300
assert raw_snapshot.shape[0] == 380, 'Raw row count differs from expected reproducible synthetic example.'

summary = pd.DataFrame({
 'stage':['Raw imported','After exact + competing duplicate resolution','After invalid/missing outcomes excluded'],
 'rows':[len(raw_snapshot),len(cleaned),len(analysis_data)]})
fig=px.bar(summary,x='stage',y='rows',color='stage',text='rows',
 title='B1. Cleaning audit trail | What changed and why',color_discrete_sequence=COLORS)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False,height=445,xaxis_title='',yaxis_title='Number of records')
fig.show()
print(f'Removed exact copies: {exact_duplicate_count}')
print(f'Resolved competing version keys: {int(conflict_keys)} (latest dated record retained)')
print(f'Invalid values set missing, by constrained field: {invalid_counter}')
print(f'Excluded rows with missing or invalid outcome: {invalid_target_rows}')
print(f'Analysis rows remaining: {len(analysis_data)} | Predictors still missing: {int(analysis_data[PREDICTORS].isna().sum().sum())}')
print('NEXT: Imputation and outlier capping will be learned from training subsets only.')

Removed exact copies: 5
Resolved competing version keys: 3 (latest dated record retained)
Invalid values set missing, by constrained field: {'men_alcohol_use_pct': 3, 'women_primary_education_pct': 1, 'women_hiv_results_received_pct': 2, 'women_age_at_first_sex_years': 3, 'synthetic_prevalence_pct': 2}
Excluded rows with missing or invalid outcome: 5
Analysis rows remaining: 367 | Predictors still missing: 105
NEXT: Imputation and outlier capping will be learned from training subsets only.


## 8. Confirm the data-quality changes interactively

This before/after comparison uses **the same analysis variables** in each chart. Hover over each bar to see how many missing cells remain. A rise in missingness after cleaning can be **correct** because impossible values have been recoded as unknown rather than silently left in the dataset.

**Key idea:** Cleaning is not about making every number look complete. It is about creating defensible, traceable records.

In [9]:
check_cols = list(BOUNDS.keys()) + ['other_indicator_03','other_indicator_07','other_indicator_11']
before = work[check_cols].isna().sum()
after = analysis_data[check_cols].isna().sum()
comp = pd.DataFrame({'variable':check_cols,'raw_missing_n':before.values,
                     'post_rule_missing_n':after.values})
long = comp.melt('variable', var_name='stage', value_name='missing_n')
fig=px.bar(long,x='variable',y='missing_n',color='stage',barmode='group',
 title='B2. Missing cells before and after rule-based cleaning',
 color_discrete_map={'raw_missing_n':BLUE,'post_rule_missing_n':CORAL})
fig.update_layout(height=500,xaxis_tickangle=-25,
                  xaxis_title='',yaxis_title='Count of missing cells')
fig.show()
print('INTERPRET: Some invalid data are now visibly missing, pending a leak-free imputation strategy.')

INTERPRET: Some invalid data are now visibly missing, pending a leak-free imputation strategy.


# PART C | Model-ready data and leakage safeguards

## 9. Split by entire *fictional regions* before estimating preprocessing

**Why grouped splitting?** Observations from the same region may be correlated. Training on some sites in a region while testing on nearby sites in the same region can inflate apparent generalization.

We reserve ~23% of the **regions**, not ~23% of randomly selected service areas, as our final test set. No fitted scaler, imputer, clipping threshold, model-selection procedure or outcome-based choice will use these held-out regions.

In [10]:
X = analysis_data[PREDICTORS].copy()
y = analysis_data['synthetic_prevalence_pct'].to_numpy(dtype=float)
groups = analysis_data['region_id'].to_numpy()
outer_split = GroupShuffleSplit(n_splits=1, test_size=.23, random_state=33)
train_idx, test_idx = next(outer_split.split(X,y,groups=groups))
X_train, X_test = X.iloc[train_idx].copy(),X.iloc[test_idx].copy()
y_train, y_test = y[train_idx],y[test_idx]
groups_train,groups_test = groups[train_idx],groups[test_idx]
assert set(groups_train).isdisjoint(set(groups_test))

split_data = pd.DataFrame({'Split':['Train','Held-out test'],
 'Service areas':[len(X_train),len(X_test)],
 'Distinct fictional regions':[len(set(groups_train)),len(set(groups_test))]})
fig=px.bar(split_data,x='Split',y='Service areas',color='Split',text='Service areas',
 title='C1. Geographic holdout | No fictional region appears in both sets',
 color_discrete_map={'Train':BLUE,'Held-out test':CORAL})
fig.update_traces(textposition='outside')
fig.update_layout(height=430,showlegend=False)
fig.show()
display(split_data)
print('Group overlap:',len(set(groups_train).intersection(set(groups_test))))
print('INTERPRET: The test regions remain untouched until the final model comparison.')

,Split,Service areas,Distinct fictional regions
0,Train,271,23
1,Held-out test,96,8


Group overlap: 0
INTERPRET: The test regions remain untouched until the final model comparison.


## 10. Treat missing predictors and plausible extremes *inside* the model pipeline

This is the **most important statistical data-cleaning safeguard** of the lesson.

**What we fit only on each training fold:**

1. `SimpleImputer(strategy="median")`: find training-fold medians and fill its predictors' missing values.
2. `QuantileClipper`: optionally learn 1st and 99th percentiles on the training fold **after imputation** and cap extreme predictors. No rows are erased just because they are unusual.
3. `StandardScaler`: learn training-fold means and standard deviations.
4. `Lasso`: estimate a sparse regression model.

**Why this order?** Computing medians, caps or scaling factors before the train/test split leaks information about unseen data. Even preprocessing within cross-validation must be refit per fold.

**Try this later:** In the setup cell, change `USE_TRAINING_ONLY_OUTLIER_CAPPING` to `False` and rerun the notebook. Compare selection and model errors. Outlier treatment is a sensitivity decision, not a universal requirement.

In [11]:
class QuantileClipper(BaseEstimator, TransformerMixin):
    """Sklearn-compatible feature-wise quantile capping, learned only on .fit(X)."""
    def __init__(self, lower=0.01, upper=0.99, enabled=True):
        self.lower, self.upper, self.enabled = lower, upper, enabled
    def fit(self, X, y=None):
        a = np.asarray(X, dtype=float)
        if not 0 <= self.lower < self.upper <= 1:
            raise ValueError('Invalid lower/upper quantiles')
        self.lower_bounds_ = np.quantile(a, self.lower, axis=0)
        self.upper_bounds_ = np.quantile(a, self.upper, axis=0)
        return self
    def transform(self, X):
        a=np.asarray(X, dtype=float)
        return np.clip(a, self.lower_bounds_, self.upper_bounds_) if self.enabled else a.copy()

def make_preprocessor():
    return [
      ('impute', SimpleImputer(strategy='median', keep_empty_features=True)),
      ('cap', QuantileClipper(lower=LOWER_CAP_QUANTILE, upper=UPPER_CAP_QUANTILE,
                               enabled=USE_TRAINING_ONLY_OUTLIER_CAPPING)),
      ('scale', StandardScaler())]

def make_lasso(alpha=0.1):
    return Pipeline(make_preprocessor() + [('lasso', Lasso(alpha=alpha, max_iter=30000))])

# Demonstrate treatment using TRAINING rows only, before all cross-validation refits.
preprocess_demo = Pipeline(make_preprocessor())
preprocess_demo.fit(X_train)
imputed = preprocess_demo.named_steps['impute'].transform(X_train)
after_cap = preprocess_demo.named_steps['cap'].transform(imputed)
view_col = 'other_indicator_01'
j = PREDICTORS.index(view_col)
comparison = pd.DataFrame({
 'value':np.r_[imputed[:,j],after_cap[:,j]],
 'version':['Training values after median imputation']*len(imputed) +
           ['Training values after optional cap']*len(after_cap)
})
fig=px.box(comparison,x='version',y='value',color='version',points='outliers',
 title=f'C2. Training-only outlier sensitivity | {view_col}',
 color_discrete_sequence=[BLUE,CORAL])
fig.update_layout(showlegend=False,height=480,xaxis_title='',yaxis_title='Feature value')
fig.show()
print(f'Mode: {"cap enabled" if USE_TRAINING_ONLY_OUTLIER_CAPPING else "cap disabled"}.')
print('INTERPRET: The charts summarize training data only. CV folds learn their OWN imputation and caps.')

Mode: cap enabled.
INTERPRET: The charts summarize training data only. CV folds learn their OWN imputation and caps.


## 11. Checkpoint: Are you ready to fit Lasso?

Before moving on, verify these five statements:

- [ ] I have preserved the untouched synthetic raw data.
- [ ] I can distinguish exact duplicates from competing record versions.
- [ ] I know why a negative percentage is invalid, but a negative signed indicator is not.
- [ ] I know why an outlier should be investigated before it is removed or capped.
- [ ] I understand why imputation, outlier caps and scaling must be learned inside each training fold.

**If any checkbox is unclear, revisit the preceding interactive chart and its interpretation.**

# PART D | Lasso regression: select variables, then evaluate

## 12. UNDERSTAND L1 regularization

Lasso minimizes the squared prediction error plus a penalty on the absolute coefficients:

\[\min_{\beta_0,\beta}\;\frac{1}{2n}\sum_{i=1}^n(y_i-\beta_0-X_i\beta)^2 + \alpha\sum_{j=1}^p|\beta_j|.\]

- **Small alpha:** weaker shrinkage, often more variables retained.
- **Large alpha:** stronger shrinkage, more coefficients exactly zero.
- **Why scale?** Variables are measured in different units, so comparing their penalized coefficients without standardization is misleading.
- **Why group CV?** The model should be evaluated on entire regions it has not encountered while fitting a fold.

The original study compared **cross-validated, minimum-BIC and adaptive Lasso**. This beginner-facing notebook uses **cross-validated Lasso and the one-standard-error rule**, not a claim that we reproduced minBIC or adaptive Lasso.

## 13. Establish a simple baseline before any advanced method

A useful model should outperform an elementary benchmark. We use a **mean-only predictor** that always forecasts the training outcome mean. Its accuracy is evaluated later on held-out regions.

In [12]:
mean_baseline = DummyRegressor(strategy='mean').fit(X_train,y_train)
print(f'Training outcome average used by baseline: {float(mean_baseline.constant_.ravel()[0]):.2f}%')
print('INTERPRET: A complex model must improve on this simple baseline on the held-out regions.')

Training outcome average used by baseline: 14.15%
INTERPRET: A complex model must improve on this simple baseline on the held-out regions.


## 14. Grouped cross-validation and conservative penalty selection

**What happens:** The training regions are split into five folds. For each candidate alpha value, the entire pipeline is fitted again, including imputation and outlier caps. We choose the **minimum CV-MSE** alpha and a more conservative **one-standard-error** alpha.

The **one-standard-error rule** selects the strongest penalty whose average CV MSE is no worse than one standard error above the lowest average CV MSE. This can reduce unstable, marginal predictors.

**Chart task:** Hover over the error curve. Why might the best alpha for prediction differ from the alpha that produces the fewest predictors?

In [13]:
inner_cv = list(GroupKFold(n_splits=5).split(X_train,y_train,groups_train))
alpha_grid=np.logspace(-2,0.5,55)
search=GridSearchCV(make_lasso(), {'lasso__alpha':alpha_grid},
                    cv=inner_cv,scoring='neg_mean_squared_error',n_jobs=-1,refit=True)
search.fit(X_train,y_train)
cv_mse=-search.cv_results_['mean_test_score']
cv_sd=search.cv_results_['std_test_score']
cv_se=cv_sd/np.sqrt(len(inner_cv))
best_pos=int(np.argmin(cv_mse))
alpha_min=float(alpha_grid[best_pos])
permitted=np.flatnonzero(cv_mse <= cv_mse[best_pos]+cv_se[best_pos])
alpha_one_se=float(max(alpha_grid[permitted]))
model_min=make_lasso(alpha_min).fit(X_train,y_train)
model_one_se=make_lasso(alpha_one_se).fit(X_train,y_train)

fig=go.Figure()
fig.add_trace(go.Scatter(x=alpha_grid,y=cv_mse+cv_se,
 line=dict(width=0),mode='lines',showlegend=False,hoverinfo='skip'))
fig.add_trace(go.Scatter(x=alpha_grid,y=cv_mse-cv_se,
 fill='tonexty',fillcolor='rgba(0,74,124,0.15)',line=dict(width=0),
 mode='lines',name='±1 fold SE',hoverinfo='skip'))
fig.add_trace(go.Scatter(x=alpha_grid,y=cv_mse,mode='lines+markers',
 line=dict(color=BLUE,width=3),marker=dict(size=5),name='Mean grouped CV MSE',
 hovertemplate='Alpha = %{x:.4f}<br>Mean CV MSE = %{y:.3f}<extra></extra>'))
fig.add_vline(x=alpha_min,line_color=TEAL,line_dash='dash',
              annotation_text=f'Min MSE: {alpha_min:.3f}',annotation_position='top left')
fig.add_vline(x=alpha_one_se,line_color=CORAL,line_dash='dot',
              annotation_text=f'1-SE: {alpha_one_se:.3f}',annotation_position='bottom right')
fig.update_xaxes(type='log',title='Lasso penalty alpha (log scale)')
fig.update_yaxes(title='Validation mean squared error (MSE)')
fig.update_layout(title='D1. Which amount of regularization works best?',height=520)
fig.show()
print(f'CV minimum-MSE alpha = {alpha_min:.4f} | one-SE alpha = {alpha_one_se:.4f}')
print('INTERPRET: One-SE favors a simpler model when differences in CV error are small.')

CV minimum-MSE alpha = 0.1292 | one-SE alpha = 0.2724
INTERPRET: One-SE favors a simpler model when differences in CV error are small.


## 15. Explore which variables Lasso retained

A Lasso coefficient **equal to zero** indicates that this fit did not retain the feature after regularization. A nonzero coefficient indicates conditional association within this synthetic model, **not causal importance**.

**Important:** Because predictors are standardized, the magnitudes here are comparable *within this fitted Lasso*. Correlated predictors can substitute for one another and make individual selections unstable.

In [14]:
coefs = pd.Series(model_one_se.named_steps['lasso'].coef_, index=PREDICTORS)
selected=coefs[coefs.abs()>1e-7].sort_values(key=lambda s:s.abs(),ascending=False)
coeff_plot = selected.sort_values()
fig=px.bar(x=coeff_plot.values,y=coeff_plot.index,orientation='h',
           color=coeff_plot.values,color_continuous_scale=['#E86852','#F4F7F9','#008D88'],
           labels={'x':'Standardized Lasso coefficient','y':'Selected indicator','color':'Coefficient'},
           title=f'D2. Conservative Lasso retained {len(selected)} of 49 candidates')
fig.update_traces(hovertemplate='%{y}<br>Standardized coefficient: %{x:+.3f}<extra></extra>')
fig.update_layout(height=max(440,140+34*len(selected)))
fig.show()
print('SELECTED synthetic teaching variables:',list(selected.index))
print('INTERPRET: A larger absolute coefficient is a stronger model-weighted association here, not proof of causality.')
print('The original Tanzania poster reported 4 Lasso-selected variables; this exercise reports its OWN result.')

SELECTED synthetic teaching variables: ['men_alcohol_use_pct', 'women_primary_education_pct', 'women_hiv_results_received_pct', 'women_age_at_first_sex_years']
INTERPRET: A larger absolute coefficient is a stronger model-weighted association here, not proof of causality.
The original Tanzania poster reported 4 Lasso-selected variables; this exercise reports its OWN result.


## 16. See how selection changes as the penalty increases

This **interactive coefficient path** shows the trade-off between simpler models and retained variables. Use the legend to isolate a signal; hover over a line to inspect how it approaches zero.

**Learning check:** When alpha is very large, what happens to most coefficients? Why can an arbitrary requirement to retain exactly four variables distort a real investigation?

In [15]:
path_alphas=np.logspace(-2,0.6,26)
coef_path=[]
for a in path_alphas:
    fit=make_lasso(float(a)).fit(X_train,y_train)
    coef_path.append(fit.named_steps['lasso'].coef_)
coef_path=np.asarray(coef_path)
fig=go.Figure()
for name in TRUE_SYNTHETIC_SIGNALS:
    j=PREDICTORS.index(name)
    fig.add_trace(go.Scatter(x=path_alphas,y=coef_path[:,j],mode='lines+markers',
      name=name,line=dict(width=3),marker=dict(size=4),
      hovertemplate=f'{name}<br>Alpha: %{{x:.4f}}<br>Coefficient: %{{y:+.3f}}<extra></extra>'))
fig.add_hline(y=0,line_dash='dash',line_color='#8D99A8')
fig.update_xaxes(type='log',title='Increasing regularization alpha →')
fig.update_yaxes(title='Coefficient after fold-safe preprocessing')
fig.update_layout(title='D3. Four planted synthetic signals along the Lasso penalty path',height=540)
fig.show()
print('INTERPRET: A predictor can disappear when regularization is stronger.')

INTERPRET: A predictor can disappear when regularization is stronger.


## 17. Check predictive performance on held-out regions

**Three comparators:**

- **Mean baseline**: Does not use any predictors.
- **Ordinary linear regression**: Uses the same imputation, capping and scaling, but no L1 penalty.
- **Conservative Lasso**: Uses grouped CV-selected alpha.

**Metrics:** RMSE and MAE are outcome-percentage-point errors here; smaller is better. R² evaluates the fraction of held-out variance explained relative to a test-set mean benchmark and can be negative. No metric is “percent accuracy.”

In [16]:
ordinary=Pipeline(make_preprocessor()+[('linear',LinearRegression())]).fit(X_train,y_train)
models={'Mean-only baseline':mean_baseline,'Ordinary regression':ordinary,
        'Lasso (1-SE)':model_one_se}
rows=[]; predictions={}
for name, model in models.items():
    pred=model.predict(X_test)
    predictions[name]=pred
    rows.append({'Model':name,'RMSE':float(np.sqrt(mean_squared_error(y_test,pred))),
                 'MAE':float(mean_absolute_error(y_test,pred)),
                 'R_squared':float(r2_score(y_test,pred))})
metrics=pd.DataFrame(rows).sort_values('RMSE')
display(metrics.round(3))
fig=px.bar(metrics,x='Model',y=['RMSE','MAE'],barmode='group',
           title='D4. Held-out geographic performance | Lower error is better',
           labels={'value':'Error (synthetic percentage points)','variable':'Error metric'},
           color_discrete_sequence=[BLUE,CORAL])
fig.update_layout(height=520)
fig.show()
print('INTERPRET: The holdout evaluates transfer to new fictional regions; it is NOT real-world Tanzania validation.')

,Model,RMSE,MAE,R_squared
1,Ordinary regression,1.546,1.217,0.830
2,Lasso (1-SE),1.556,1.232,0.828
0,Mean-only baseline,3.775,3.179,-0.012


INTERPRET: The holdout evaluates transfer to new fictional regions; it is NOT real-world Tanzania validation.


## 18. Explore predictions versus observations

Use hover to inspect test records. The diagonal represents perfect prediction. Points far from it have larger errors, which may reflect omitted variables, noise, small-area differences or model limitations.

**Ask:** Does the fit look equally good across the low and high ends of the synthetic outcome?

In [17]:
plot_df=pd.DataFrame({'Observed outcome (%)':y_test,
 'Predicted outcome (%)':predictions['Lasso (1-SE)'],
 'Fictional region':groups_test,
 'Absolute error':np.abs(y_test-predictions['Lasso (1-SE)'])})
fig=px.scatter(plot_df,x='Observed outcome (%)',y='Predicted outcome (%)',
  color='Absolute error',hover_data=['Fictional region'],
  color_continuous_scale='Viridis',opacity=.87,
  title='D5. Lasso predictions on unseen fictional regions')
lo=float(min(y_test.min(),plot_df['Predicted outcome (%)'].min()))
hi=float(max(y_test.max(),plot_df['Predicted outcome (%)'].max()))
fig.add_trace(go.Scatter(x=[lo,hi],y=[lo,hi],mode='lines',name='Perfect agreement',
                         line=dict(color=CORAL,dash='dash',width=2)))
fig.update_layout(height=550)
fig.show()
print('INTERPRET: Closer to the diagonal means a smaller prediction error, not a causal association.')

INTERPRET: Closer to the diagonal means a smaller prediction error, not a causal association.


## 19. Selection stability: are the same predictors chosen repeatedly?

We repeatedly resample **whole training regions** and refit the same preprocessor and Lasso penalty. The percentage of fits retaining a predictor is called its **selection frequency**.

**Interpretation warning:** Selection frequency is a practical sensitivity diagnostic. It is **not** a confidence interval, p-value or causal probability.

In [18]:
stability_split=GroupShuffleSplit(n_splits=30,test_size=.20,random_state=814)
selection_flags=[]
for sub_idx,_ in stability_split.split(X_train,y_train,groups_train):
    fit=make_lasso(alpha_one_se).fit(X_train.iloc[sub_idx],y_train[sub_idx])
    selection_flags.append(np.abs(fit.named_steps['lasso'].coef_)>1e-7)
stability=pd.Series(np.mean(selection_flags,axis=0)*100,index=PREDICTORS)
frequency=stability.sort_values(ascending=False).head(12).sort_values()
fig=px.bar(x=frequency.values,y=frequency.index,orientation='h',
           title='D6. How often does each variable survive region resampling?',
           labels={'x':'Selection frequency over 30 fits (%)','y':'Candidate indicator'},
           color=frequency.values,color_continuous_scale='Tealgrn')
fig.update_xaxes(range=[0,105])
fig.update_layout(height=530,coloraxis_showscale=False)
fig.show()
print('INTERPRET: Variables selected in very few resamples deserve caution and further validation.')

INTERPRET: Variables selected in very few resamples deserve caution and further validation.


# PART E | Prepare for GWR and public health interpretation

## 20. Check correlations among the study-inspired indicators

Multicollinearity can complicate both variable selection and downstream spatial models. This heatmap is an **initial screen**, not a complete GWR assumption check.

**Research context:** The actual poster excluded women's age at first sexual intercourse from GWR because of multicollinearity. We do **not** artificially require the synthetic data to show that same correlation.

In [19]:
corr=X_train[TRUE_SYNTHETIC_SIGNALS].corr()
fig=px.imshow(corr,text_auto='.2f',zmin=-1,zmax=1,
              color_continuous_scale='RdBu_r',
              title='E1. Correlation screen | Four fictional study-inspired indicators',
              labels={'color':'Pearson r'})
fig.update_layout(height=565)
fig.show()
print('INTERPRET: Correlations near +1 or -1 warrant investigation; correlation alone does not establish causation.')

INTERPRET: Correlations near +1 or -1 warrant investigation; correlation alone does not establish causation.


## 21. Export a **fully synthetic** spatial-analysis handoff

The handoff summarizes the cleaned teaching data by fictional region and supplies invented planar coordinates. It preserves the study-inspired indicator concepts so that the next lesson can explore GWR methodology.

**What is deliberately missing:** There is no genuine Tanzania boundary, no official region name, and no spatially varying GWR coefficient. Do not plot these invented coordinates on a real map.

**Notebook users:** The CSV is generated when you run the cell. In Google Colab, open the Files pane to download it if desired. No upload is required.

In [20]:
spatial_cols=['fictional_x','fictional_y','synthetic_prevalence_pct']+TRUE_SYNTHETIC_SIGNALS
handoff=analysis_data.groupby('region_id',as_index=False)[spatial_cols].mean(numeric_only=True)
handoff['DATA_PROVENANCE']='FULLY_SYNTHETIC_TEACHING_DATA_NOT_TANZANIA'
assert len(handoff)==31
filename='EvidenceLab11_GWR_Handoff_FULLY_SYNTHETIC.csv'
handoff.to_csv(filename,index=False)
fig=px.scatter(handoff,x='fictional_x',y='fictional_y',
               size='synthetic_prevalence_pct',color='synthetic_prevalence_pct',
               hover_name='region_id',
               hover_data={'synthetic_prevalence_pct':':.2f'},
               color_continuous_scale='Viridis',
               title='E2. Invented coordinates | An illustration, NOT a real Tanzania map')
fig.update_layout(height=535)
fig.show()
print(f'Created {filename} with {len(handoff)} fictional regions.')
print('INTERPRET: Marker size shows the simulated outcome; no local regression has been estimated yet.')

Created EvidenceLab11_GWR_Handoff_FULLY_SYNTHETIC.csv with 31 fictional regions.
INTERPRET: Marker size shows the simulated outcome; no local regression has been estimated yet.


## 22. Consolidated learning review

**What you just did:**

1. **Data provenance:** Built 49 synthetic predictors and a synthetic outcome across 31 fictional regions.
2. **Missingness:** Quantified and mapped missing values, including hidden text placeholders.
3. **Duplicates:** Distinguished exact copies from competing versions and recorded your resolution rule.
4. **Negative values and ranges:** Applied domain-specific constraints without deleting valid signed indicators.
5. **Outliers:** Flagged suspicious values and tried fold-safe, optional quantile capping rather than automatic deletion.
6. **Leakage control:** Split entire regions, and trained imputers, caps, scalers and Lasso inside cross-validation folds.
7. **Selection:** Used grouped CV and the one-standard-error rule to identify a manageable subset of variables.
8. **Validation:** Compared against baseline and ordinary regression on unseen fictional regions and tested selection stability.
9. **Spatial handoff:** Created a synthetic file for the upcoming GWR demonstration.

### What the Tanzania study actually reported

From 49 candidate predictors, both CV and minBIC Lasso highlighted **men's alcohol consumption, women's completion of primary education, women's age at first sexual intercourse, and women receiving HIV test results**. The GWR analyses examined three indicators after excluding the age-at-first-sex indicator for multicollinearity. The poster described regionally differing associations, including **men's alcohol use in the northeast, women's primary education in the southeast, and HIV test-result receipt among women in the southwest**. The poster's adjusted GWR R² was **0.784**. Those are reported poster results, **not outcomes of this notebook**.

**Keep uncertainty front and center:** A local association does not establish a mechanism, a causal driver or the benefit of an intervention. Disease surveillance and outbreak investigation require context-specific outcome definitions, appropriate data units, validation, spatial diagnostics, careful interpretation and field verification.

### Next: EvidenceLab #12

**GWR: What drives the hotspot?** We will ask a more precise question: **How do the estimated associations between selected indicators and outcomes vary across places?** We will explore bandwidth choices, local coefficients, multiple-testing concerns and mapping uncertainty.

**Follow Dr. Amobi Andrew Onovo on [LinkedIn](https://www.linkedin.com/in/amobi-onovo-ph-d-global-health-pgdip-data-science-mph-epidemiology-135a1025/) and subscribe to the EvidenceLab YouTube series** for the GWR lesson.

**EvidenceLab | See it. Understand it. Run it.**

### Self-check quiz (answers included for independent learners)

**Q1.** If an indicator is missing in 6% of records, should you replace each missing value with zero?  
**A1.** Not automatically. Zero may be a meaningful measurement; missing means unknown. Investigate the mechanism and apply an appropriate model-specific treatment.

**Q2.** Should you delete every negative number in a health dataset?  
**A2.** No. Apply variable-specific definitions. A negative percentage is invalid; a negative standardized score may be valid.

**Q3.** What is the difference between an outlier flag and an invalid value?  
**A3.** An outlier is unusual under a distributional rule and deserves review; an invalid value violates an established data rule.

**Q4.** Why not fill missing predictors with the median of the entire dataset before cross-validation?  
**A4.** That allows held-out validation/test observations to influence preprocessing parameters, causing information leakage.

**Q5.** Does Lasso prove that a selected variable caused HIV acquisition?  
**A5.** No. Lasso performs penalized variable selection and prediction/association modeling, not causal identification.

**Q6.** Why can't we label the notebook's simulated charts as Tanzania study results?  
**A6.** The notebook generates synthetic records and arbitrary associations; it does not contain actual survey observations or reproduce the original study's design/estimates.

**Source:** Onovo et al., AIDS 2024, poster TUPED353. https://quantiuminsights.com/research.html